# Forbes Global 2000 (2026) — Exploratory Data Analysis

This notebook explores the cleaned Forbes Global 2000 dataset before/alongside the interactive dashboard. It documents the same findings used in the dashboard's Insights panel, with the underlying pandas code visible.

Data: `data/processed/forbes_2000_cleaned.csv` (see `scripts/clean_forbes_2000.py` for the cleaning pipeline).

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/processed/forbes_2000_cleaned.csv')
df.shape

## 1. Quick look at the data

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

## 2. Geographic distribution

Which countries have the most companies in the Global 2000?

In [ ]:
top_countries = df['Country'].value_counts().head(15)
top_countries

In [ ]:
top_countries.plot(kind='barh', figsize=(8,6), title='Top 15 Countries by Company Count').invert_yaxis()

## 3. Industry breakdown

In [ ]:
industry_counts = df['Industry'].value_counts()
industry_counts.head(10)

In [ ]:
industry_counts.head(10).plot(kind='barh', figsize=(8,6), title='Top 10 Industries by Company Count').invert_yaxis()

## 4. Market value concentration

How much of total market value sits in the top 10 companies?

In [ ]:
with_mv = df.dropna(subset=['Market Value ($B)'])
top10_mv = with_mv.nlargest(10, 'Market Value ($B)')
concentration = top10_mv['Market Value ($B)'].sum() / with_mv['Market Value ($B)'].sum() * 100
print(f"Top 10 companies hold {concentration:.1f}% of total market value")
top10_mv[['Rank','Company','Country','Market Value ($B)']]

## 5. Profitability: margin and ROA distributions

In [ ]:
df['Profit_Margin_%'].describe()

In [ ]:
df['Profit_Margin_%'].clip(upper=100).hist(bins=30, figsize=(8,5))
# clipped at 100% for readability — a handful of extreme outliers (e.g. Broadcom) skew the raw histogram

In [ ]:
df['ROA_%'].describe()

## 6. Market value by rank tier

The same rank-bracket aggregation used for the dashboard's trend chart.

In [ ]:
df['rank_bracket'] = ((df['Rank']-1)//100)*100
bracket_agg = df.groupby('rank_bracket').agg({'Market Value ($B)':'sum','Sales ($B)':'sum'}).reset_index()
bracket_agg

In [ ]:
bracket_agg.plot(x='rank_bracket', y='Market Value ($B)', figsize=(8,5), title='Combined Market Value by Rank Bracket', legend=False)

## 7. Industry-level profitability leaders

Which industries lead on average profit margin vs. average ROA? (Restricted to industries with at least 5 companies to reduce noise.)

In [ ]:
ind_stats = df.groupby('Industry').agg(
    company_count=('Company','count'),
    avg_margin=('Profit_Margin_%','mean'),
    avg_roa=('ROA_%','mean')
).query('company_count >= 5')

print('Top 5 industries by average profit margin:')
display(ind_stats.sort_values('avg_margin', ascending=False).head())

print('Top 5 industries by average ROA:')
display(ind_stats.sort_values('avg_roa', ascending=False).head())

## 8. Summary

These are the same headline findings surfaced dynamically in the dashboard's Insights & Recommendations panel — recomputed live there for whatever subset of companies a user has filtered to. Here they're shown once, for the full dataset:

- Market value is concentrated in a small number of mega-cap companies
- The US, China, and Japan dominate by company count
- Profitability varies widely by industry — some sectors post very high margins on relatively low ROA (asset-heavy businesses like banking), while others are the reverse